In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import requests
from io import BytesIO
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

In [ ]:
# Image size
imsize = 512 if torch.cuda.is_available() else 256

loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor()
])

def load_image_from_url(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    response = requests.get(url, headers=headers)

    # Check if the response content is actually an image
    if not response.headers.get('Content-Type', '').startswith('image'):
        raise ValueError(f"URL did not return an image: {url} (Content-Type: {response.headers.get('Content-Type')})")

    img = Image.open(BytesIO(response.content)).convert('RGB')
    img = loader(img).unsqueeze(0)
    return img.to(device, torch.float)

def show_image(tensor, title=None):
    image = tensor.cpu().clone()
    image = image.squeeze(0)
    image = transforms.ToPILImage()(image)
    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis('off')

# Load content image (a photo)
content_url = "https://i.imgur.com/FwYhWJ2.jpeg"

# Load style image (Van Gogh's Starry Night)
style_url = "https://i.imgur.com/K5bZ0rW.jpeg"

content_img = load_image_from_url(content_url)
style_img = load_image_from_url(style_url)

# Show original images
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plt.sca(axes[0])
show_image(content_img, "Content Image")
plt.sca(axes[1])
show_image(style_img, "Style Image (Van Gogh)")
plt.tight_layout()
plt.savefig("original_images.png")
plt.show()
print("✅ Images loaded successfully!")

In [ ]:
# Load pretrained VGG19
vgg = models.vgg19(pretrained=True).features.to(device).eval()

# Freeze parameters
for param in vgg.parameters():
    param.requires_grad_(False)

print("✅ VGG19 loaded!")

# Normalization for VGG
class Normalization(nn.Module):
    def __init__(self):
        super().__init__()
        mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).to(device)
        self.mean = mean.view(-1, 1, 1)
        self.std = std.view(-1, 1, 1)

    def forward(self, img):
        return (img - self.mean) / self.std

# Loss classes
class ContentLoss(nn.Module):
    def __init__(self, target):
        super().__init__()
        self.target = target.detach()

    def forward(self, input):
        self.loss = nn.functional.mse_loss(input, self.target)
        return input

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super().__init__()
        self.target = self.gram_matrix(target_feature).detach()

    def gram_matrix(self, input):
        b, c, h, w = input.size()
        features = input.view(b * c, h * w)
        G = torch.mm(features, features.t())
        return G.div(b * c * h * w)

    def forward(self, input):
        G = self.gram_matrix(input)
        self.loss = nn.functional.mse_loss(G, self.target)
        return input

print("✅ Loss functions defined!")

In [ ]:
def build_style_transfer_model(vgg, content_img, style_img):
    content_layers = ['conv_4']
    style_layers = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

    content_losses = []
    style_losses = []

    normalization = Normalization().to(device)
    model = nn.Sequential(normalization)

    i = 0
    for layer in vgg.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f'conv_{i}'
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f'pool_{i}'
        elif isinstance(layer, nn.BatchNorm2d):
            name = f'bn_{i}'
        else:
            continue

        model.add_module(name, layer)

        if name in content_layers:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module(f"content_loss_{i}", content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            target = model(style_img).detach()
            style_loss = StyleLoss(target)
            model.add_module(f"style_loss_{i}", style_loss)
            style_losses.append(style_loss)

    # Trim model after last loss layer
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], (ContentLoss, StyleLoss)):
            break
    model = model[:(i + 1)]

    return model, content_losses, style_losses

model, content_losses, style_losses = build_style_transfer_model(
    vgg, content_img, style_img
)
print("✅ Style Transfer model built!")

In [ ]:
def run_style_transfer(content_img, style_img,
                        num_steps=200,
                        style_weight=1000000,
                        content_weight=1):

    print("🎨 Running Neural Style Transfer...")

    # Start with content image
    input_img = content_img.clone()
    optimizer = optim.LBFGS([input_img.requires_grad_(True)])

    model, content_losses, style_losses = build_style_transfer_model(
        vgg, content_img, style_img
    )

    step = [0]
    while step[0] <= num_steps:
        def closure():
            input_img.data.clamp_(0, 1)
            optimizer.zero_grad()
            model(input_img)

            style_score = sum(sl.loss for sl in style_losses) * style_weight
            content_score = sum(cl.loss for cl in content_losses) * content_weight

            loss = style_score + content_score
            loss.backward()

            step[0] += 1
            if step[0] % 50 == 0:
                print(f"Step {step[0]}: "
                      f"Style Loss: {style_score.item():.2f} "
                      f"Content Loss: {content_score.item():.2f}")
            return loss

        optimizer.step(closure)

    input_img.data.clamp_(0, 1)
    return input_img

# Run it!
output = run_style_transfer(content_img, style_img, num_steps=200)
print("✅ Style Transfer complete!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

plt.sca(axes[0])
show_image(content_img, "Content Image")

plt.sca(axes[1])
show_image(style_img, "Style Image\n(Van Gogh - Starry Night)")

plt.sca(axes[2])
show_image(output, "Stylized Output 🎨")

plt.tight_layout()
plt.savefig("neural_style_transfer_result.png", dpi=150)
plt.show()
print("✅ Final result saved!")